# REMD analysis — parallel tempering (openmmtools)

Explore the output of `remd_openmmtools.py` (8 replicas, 300–500 K geometric ladder).

The simulation stores everything in a single NetCDF (`out/remd_remd.nc`) via
openmmtools' `MultiStateReporter`. This notebook reads it and walks through the
standard REMD diagnostics:

1. **Run summary** — ladder, iterations, steps.
2. **Exchange mixing** — neighbour acceptance + full transition matrix (does the ladder mix?).
3. **Walker diffusion** — how each replica random-walks through temperature space (round trips).
4. **Energy overlap** — potential-energy distributions per temperature (why exchanges do/don't accept).
5. **Free energy** — MBAR relative free energies of the temperature states.
6. **Structural exploration** — demultiplex the 300 K ensemble to a trajectory and look at Rg.

Everything is read-only; rerun any cell freely.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from openmm import unit
from openmmtools.multistate import MultiStateReporter, ParallelTemperingAnalyzer

NC   = "out/remd_remd.nc"      # REMD storage
PSF  = "out/remd.psf"          # topology (for the structural cell)

reporter = MultiStateReporter(NC, open_mode="r")
analyzer = ParallelTemperingAnalyzer(reporter)

# Temperature ladder (state index -> temperature in K)
states = reporter.read_thermodynamic_states()[0]
temps  = np.array([s.temperature.value_in_unit(unit.kelvin) for s in states])
n_states = len(temps)

# Molar Boltzmann constant, so beta = 1/(KB_NA * T) pairs with energies in kJ/mol
KB_NA = (unit.BOLTZMANN_CONSTANT_kB * unit.AVOGADRO_CONSTANT_NA).value_in_unit(
    unit.kilojoule_per_mole / unit.kelvin)
betas = 1.0 / (KB_NA * temps)

print(f"{n_states} replicas / temperature states")
print("Ladder (K):", "  ".join(f"{t:.1f}" for t in temps))

## 1. Run summary

`read_replica_thermodynamic_states()` returns a `(n_iterations+1, n_replicas)` array:
entry `[i, r]` is the temperature-**state index** occupied by replica `r` at iteration `i`.
`read_energies()` returns the reduced-potential matrix `u[i, r, k]` — replica `r`'s
configuration at iteration `i`, evaluated in state `k` (units of kT).

In [ ]:
sst = np.asarray(reporter.read_replica_thermodynamic_states())   # (n_iter, n_replicas)
u_irk = np.asarray(reporter.read_energies()[0])                  # (n_iter, n_replicas, n_states)
n_iter = sst.shape[0]

print(f"iterations stored : {n_iter}  (incl. iteration 0)")
print(f"replicas          : {sst.shape[1]}")
print(f"reduced-energy matrix shape u[iter, replica, state] : {u_irk.shape}")

## 2. Exchange mixing

`generate_mixing_statistics()` gives the empirical **state-transition matrix** `T[i,j]`
(probability a replica in state `i` moves to state `j` at an exchange attempt). The
super-/sub-diagonal are the neighbour acceptance rates — the numbers you tune the ladder on.

**Rule of thumb:** aim for neighbour acceptance ≈ 0.2–0.4 across the whole ladder. Much
lower and neighbouring states barely swap (a bottleneck); much higher and you are wasting
replicas on too-fine spacing.

In [ ]:
stats = analyzer.generate_mixing_statistics()
Tij = np.asarray(getattr(stats, "transition_matrix", stats[0]))

neighbor_acc = np.array([Tij[i, i + 1] for i in range(n_states - 1)])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))

im = ax0.imshow(Tij, cmap="viridis", vmin=0, vmax=Tij.max())
ax0.set(title="State transition matrix  T[i, j]", xlabel="to state j", ylabel="from state i")
ax0.set_xticks(range(n_states)); ax0.set_yticks(range(n_states))
fig.colorbar(im, ax=ax0, fraction=0.046)

pairs = [f"{temps[i]:.0f}\u2013{temps[i+1]:.0f}" for i in range(n_states - 1)]
bars = ax1.bar(pairs, neighbor_acc, color="#4C78A8")
ax1.axhspan(0.2, 0.4, color="green", alpha=0.12, label="healthy 0.2\u20130.4")
ax1.set(title="Neighbour exchange acceptance", ylabel="acceptance", ylim=(0, max(0.5, neighbor_acc.max()*1.1)))
ax1.set_xticklabels(pairs, rotation=45, ha="right")
ax1.legend()
for b, v in zip(bars, neighbor_acc):
    ax1.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

worst = int(np.argmin(neighbor_acc))
print(f"weakest pair: {temps[worst]:.0f}-{temps[worst+1]:.0f} K  (acc={neighbor_acc[worst]:.3f})"
      f"  -> add replicas / narrow spacing here if < 0.2")

## 3. Walker diffusion through temperature space

For good sampling a replica should *random-walk* up and down the whole ladder — a
"round trip" (bottom → top → bottom) carries a low-temperature structure up to high
temperature to unfold and back down to refold, which is the entire point of PT.

The left plot traces a few replicas' state index over time. The right plot shows how
evenly each **state** was occupied (should be ~flat = uniform).

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))

for r in range(min(4, sst.shape[1])):
    ax0.plot(sst[:, r], lw=1, alpha=0.8, label=f"replica {r}")
ax0.set(title="Walker trajectory in state space", xlabel="iteration",
        ylabel="temperature-state index", yticks=range(n_states))
ax0.set_yticklabels([f"{t:.0f}" for t in temps])
ax0.legend(fontsize=8, ncol=2)

occ = np.array([(sst == k).sum() for k in range(n_states)]) / sst.size
ax1.bar([f"{t:.0f}" for t in temps], occ, color="#F58518")
ax1.axhline(1/n_states, ls="--", c="k", lw=1, label="uniform")
ax1.set(title="State occupancy (fraction of replica-iterations)", xlabel="temperature (K)",
        ylabel="occupancy"); ax1.legend()
plt.tight_layout(); plt.show()

# crude round-trip count: how many times each replica touches both extremes in sequence
def round_trips(track):
    trips, target = 0, 0  # 0 -> looking to reach bottom(0); toggles
    at = None
    for s in track:
        if s == 0: at = "bottom"
        elif s == n_states - 1:
            if at == "bottom": trips += 1
            at = "top"
    return trips
print("round trips per replica (bottom->top):",
      [round_trips(sst[:, r]) for r in range(sst.shape[1])])

## 4. Potential-energy overlap between temperatures

Two neighbouring temperatures can only exchange if their **potential-energy distributions
overlap** — the acceptance rate in §2 is essentially the size of that overlap. Here we
demultiplex the reduced energies back into physical potential energy `U` (kJ/mol) for the
ensemble actually sampled at each temperature, and plot the histograms.

Well-tuned ladders show adjacent histograms overlapping by a good fraction; a gap between
two curves is exactly where acceptance collapses.

In [ ]:
# U of the config in replica r at iter i (its CURRENT state s): u[i,r,s] = beta_s * U  ->  U = u/beta_s
U_by_state = {k: [] for k in range(n_states)}
for i in range(n_iter):
    for r in range(sst.shape[1]):
        s = sst[i, r]
        U_by_state[s].append(u_irk[i, r, s] / betas[s])

plt.figure(figsize=(9, 5))
cmap = plt.cm.coolwarm(np.linspace(0, 1, n_states))
for k in range(n_states):
    U = np.array(U_by_state[k])
    plt.hist(U, bins=30, density=True, histtype="stepfilled", alpha=0.35,
             color=cmap[k], label=f"{temps[k]:.0f} K")
plt.xlabel("potential energy  U  (kJ/mol)"); plt.ylabel("probability density")
plt.title("Potential-energy distributions per temperature (overlap = exchange feasibility)")
plt.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

## 5. MBAR relative free energies

`get_free_energy()` runs MBAR over all states and returns the dimensionless free-energy
differences `f_k` (in kT) with statistical errors. This is the thermodynamic payoff of the
run — combined with the potential energies it yields heat capacity, etc.

*(With only ~100 exchange attempts this is illustrative, not converged.)*

In [ ]:
F, dF = analyzer.get_free_energy()      # (n_states, n_states) matrices, units of kT
f = F[0] - F[0, 0]                       # relative to the 300 K state
ferr = dF[0]

plt.figure(figsize=(7, 4.5))
plt.errorbar(temps, f, yerr=ferr, marker="o", capsize=4, color="#54A24B")
plt.xlabel("temperature (K)"); plt.ylabel(r"$f_k - f_0$  (kT)")
plt.title("Relative dimensionless free energy vs temperature")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for t, v, e in zip(temps, f, ferr):
    print(f"  {t:6.1f} K :  {v:8.3f} +/- {e:.3f}  kT")

## 6. Structural exploration — demultiplex the 300 K ensemble

Each output file mixes all temperatures, so to *look at structures at one temperature* you
demultiplex: pull the configuration of whichever replica currently sits at the target state.

**Note:** openmmtools stores full coordinates only every `checkpoint_interval` iterations
(here every 10), so the structural ensemble is coarser than the energy/state records above.
Below we extract the **300 K (state 0)** ensemble at those checkpoint frames into an mdtraj
trajectory and plot the radius of gyration — a quick folded/unfolded readout. Change
`TARGET_STATE` to explore any rung.

In [ ]:
import mdtraj as md

TARGET_STATE = 0   # 0 = coldest (300 K); n_states-1 = hottest

# Coordinates are stored only at checkpoint iterations (every checkpoint_interval).
ckpt = reporter.checkpoint_interval
xyz, frame_iters = [], []
for i in range(0, n_iter, ckpt):
    sampler_states = reporter.read_sampler_states(iteration=i)
    if sampler_states is None:      # defensive: skip iterations without stored positions
        continue
    r = int(np.where(sst[i] == TARGET_STATE)[0][0])   # replica at the target temperature
    xyz.append(sampler_states[r].positions.value_in_unit(unit.nanometer))
    frame_iters.append(i)
xyz = np.asarray(xyz)

top = md.load_psf(PSF)
traj = md.Trajectory(xyz, top)
print(f"demuxed {traj.n_frames} checkpoint frames at {temps[TARGET_STATE]:.0f} K "
      f"(iterations {frame_iters}), {traj.n_atoms} CA atoms")

# Optionally persist for external analysis / visualization:
# traj.save_dcd(f"out/demux_state{TARGET_STATE}.dcd")

rg = md.compute_rg(traj)   # nm
plt.figure(figsize=(9, 4))
plt.plot(frame_iters, rg * 10, marker="o", lw=1, color="#B279A2")   # nm -> Angstrom
plt.axhline(rg.mean() * 10, ls="--", c="k", lw=1, label=f"mean {rg.mean()*10:.2f} A")
plt.xlabel("iteration"); plt.ylabel("radius of gyration (A)")
plt.title(f"Rg of the {temps[TARGET_STATE]:.0f} K ensemble"); plt.legend()
plt.tight_layout(); plt.show()

---
### Notes & next steps

- **Ladder tuning:** if any neighbour acceptance in §2 drops below ~0.2, insert replicas
  there (or lower `--tmax`) and rerun. In this short demo the top of the ladder
  (≈465–500 K) is the bottleneck.
- **Convergence:** 100 exchange attempts is far too short for the free energies in §5 —
  raise `--cycles` for production and check that §3 shows several full round trips.
- **Restart:** the run wrote `out/remd_remd_checkpoint.nc`; `sampler.run()` can be resumed
  by re-opening the same storage (not covered here).
- **More structure:** uncomment `traj.save_dcd(...)` in §6 to export any temperature's
  ensemble for RMSD/contact/secondary-structure analysis with your usual tools.

In [ ]:
reporter.close()